# Notebook 03 — Modélisation : Prédiction du Churn

Quatre approches sont comparées pour prédire la probabilité qu'un client parte dans les prochains mois :

1. **Régression Logistique** — point de départ interprétable, référence pour les gains des modèles suivants
2. **Random Forest** — capture les non-linéarités, fournit une importance native des variables
3. **XGBoost** — gradient boosting, souvent performant sur des données tabulaires
4. **MLP (réseau de neurones)** — test du deep learning sur ce volume de données

Sur des données tabulaires de 10 000 lignes avec 10% de déséquilibre, la métrique à surveiller est le **F1-score** et la **PR-AUC** — l'accuracy seule est trompeuse (un modèle qui prédit toujours "non-churner" aurait 90% d'accuracy).

In [1]:
import sys
sys.path.append('../src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import xgboost as xgb
import tensorflow as tf
from tensorflow import keras

from evaluation import (
    classification_metrics, compare_classifiers,
    plot_confusion_matrix, error_analysis_classification,
)

# Charger les données prétraitées (issues du notebook 02)
data = joblib.load('../models/processed_data.joblib')
X_train, X_test   = data['X_train'], data['X_test']
y_train, y_test   = data['y_train'], data['y_test']
feature_names     = data['feature_names']
X_test_raw        = data['X_test_raw']

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Churn rate train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')

X_train: (8000, 50) | X_test: (2000, 50)
Churn rate train: 0.102 | test: 0.102


## 1. Baseline — Régression Logistique

In [2]:
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

y_pred_lr  = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

lr_metrics = classification_metrics(y_test, y_pred_lr, y_proba_lr, 'Logistic Regression')
print(lr_metrics)
plot_confusion_matrix(y_test, y_pred_lr, 'Logistic Regression', '../reports/figures/03_cm_lr.png')

{'model': 'Logistic Regression', 'accuracy': 0.697, 'precision': 0.20266272189349113, 'recall': 0.6715686274509803, 'f1': 0.31136363636363634, 'roc_auc': 0.7478983798419145, 'pr_auc': 0.2707798397720261}


<Figure size 500x400 with 1 Axes>

## 2. Random Forest

In [3]:
rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

rf_metrics = classification_metrics(y_test, y_pred_rf, y_proba_rf, 'Random Forest')
print(rf_metrics)

# Feature importance native (niveau basique — Permutation et SHAP dans notebook 05)
fi = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
fi.head(12).plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Random Forest — Feature Importance native (top 12)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/03_rf_feature_importance.png', bbox_inches='tight')
plt.show()

{'model': 'Random Forest', 'accuracy': 0.887, 'precision': 0.35526315789473684, 'recall': 0.1323529411764706, 'f1': 0.19285714285714287, 'roc_auc': 0.8022989541027993, 'pr_auc': 0.27121740316034304}


C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_10880\3524911560.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. XGBoost

In [4]:
scale_pw = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=200, scale_pos_weight=scale_pw,
    random_state=42, eval_metric='logloss', verbosity=0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False,
)

y_pred_xgb  = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
xgb_metrics = classification_metrics(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost')
print(xgb_metrics)

{'model': 'XGBoost', 'accuracy': 0.872, 'precision': 0.25471698113207547, 'recall': 0.1323529411764706, 'f1': 0.17419354838709677, 'roc_auc': 0.7409807742696188, 'pr_auc': 0.21859080122081936}


## 4. MLP (Deep Learning)

**Justification de l'architecture** : 2 couches cachées (128, 64 neurones) avec Dropout pour limiter l'overfitting, BatchNormalization pour la stabilité de l'entraînement. Early stopping sur la validation loss (patience=10) pour éviter le sur-apprentissage.

In [5]:
tf.random.set_seed(42)

mlp = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid'),
])
mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['AUC'])

history = mlp.fit(
    X_train, y_train,
    validation_split=0.15,
    epochs=100, batch_size=64,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)],
    class_weight={0: 1.0, 1: scale_pw},  # équilibrage des classes
    verbose=0,
)

print(f'Epochs effectuées : {len(history.history["loss"])}')

# Courbes d'apprentissage
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history.history['loss'], label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Binary Crossentropy')
ax.set_title('MLP — Courbes d\'apprentissage (Early Stopping)')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/03_mlp_learning_curves.png', bbox_inches='tight')
plt.show()

Epochs effectuées : 37


C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_10880\473568663.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
y_proba_mlp = mlp.predict(X_test, verbose=0).flatten()
y_pred_mlp  = (y_proba_mlp >= 0.5).astype(int)
mlp_metrics = classification_metrics(y_test, y_pred_mlp, y_proba_mlp, 'MLP')
print(mlp_metrics)

{'model': 'MLP', 'accuracy': 0.786, 'precision': 0.24545454545454545, 'recall': 0.5294117647058824, 'f1': 0.33540372670807456, 'roc_auc': 0.742161229748024, 'pr_auc': 0.21883200253744584}


## 5. Comparaison structurée — tableau et visualisations

In [7]:
all_metrics = [lr_metrics, rf_metrics, xgb_metrics, mlp_metrics]
comparison  = compare_classifiers(all_metrics)
print('=== Tableau comparatif — Tâche A (churn) ===')
display(comparison)

=== Tableau comparatif — Tâche A (churn) ===


,accuracy,precision,recall,f1,roc_auc,pr_auc
model,,,,,,
MLP,0.786,0.2455,0.5294,0.3354,0.7422,0.2188
Logistic Regression,0.697,0.2027,0.6716,0.3114,0.7479,0.2708
Random Forest,0.887,0.3553,0.1324,0.1929,0.8023,0.2712
XGBoost,0.872,0.2547,0.1324,0.1742,0.7410,0.2186


In [8]:
# Radar chart multi-modèles
import plotly.graph_objects as go

metrics_radar = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']
metrics_radar = [m for m in metrics_radar if m in comparison.columns]

fig = go.Figure()
for model_name in comparison.index:
    values = comparison.loc[model_name, metrics_radar].tolist()
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=metrics_radar + [metrics_radar[0]],
        fill='toself', name=model_name,
    ))
fig.update_layout(polar=dict(radialaxis=dict(range=[0, 1])),
                  title='Comparaison radar — 4 modèles (test set)')
fig.show()
fig.write_html('../reports/figures/03_radar_comparison.html')

In [9]:
# Courbes ROC
from sklearn.metrics import roc_curve, precision_recall_curve

models_proba = {
    'Logistic Regression': y_proba_lr,
    'Random Forest': y_proba_rf,
    'XGBoost': y_proba_xgb,
    'MLP': y_proba_mlp,
}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, proba in models_proba.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=name)
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_title('Courbes ROC')
axes[0].set_xlabel('Taux de faux positifs')
axes[0].set_ylabel('Taux de vrais positifs')
axes[0].legend()

for name, proba in models_proba.items():
    prec, rec, _ = precision_recall_curve(y_test, proba)
    axes[1].plot(rec, prec, label=name)
axes[1].set_title('Courbes Precision-Recall (PR-AUC)')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/03_roc_pr_curves.png', bbox_inches='tight')
plt.show()

C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_10880\2468602229.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Validation croisée (stabilité)

In [10]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {'Logistic Regression': lr, 'Random Forest': rf, 'XGBoost': xgb_model}

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring='f1', n_jobs=-1)
    cv_results[name] = {'mean_f1': scores.mean(), 'std_f1': scores.std()}
    print(f'{name}: F1 = {scores.mean():.4f} ± {scores.std():.4f}')

print('\n→ Un écart-type faible indique un modèle stable (ne dépend pas du split).')

Logistic Regression: F1 = 0.3199 ± 0.0120


Random Forest: F1 = 0.1558 ± 0.0370


XGBoost: F1 = 0.1923 ± 0.0273

→ Un écart-type faible indique un modèle stable (ne dépend pas du split).


## 7. Analyse d'erreurs — Faux Négatifs (churners manqués)

In [11]:
# Utiliser le meilleur modèle (à ajuster selon les résultats)
best_name  = comparison.index[0]
best_proba = models_proba[best_name]
best_pred  = (best_proba >= 0.5).astype(int)

fn_analysis = error_analysis_classification(
    X_test_raw.reset_index(drop=True), y_test, best_pred, best_proba, n_worst=10
)
print(f'Top 10 churners manqués par {best_name} (faux négatifs) :')
display(fn_analysis)
print('\n→ Ces clients ont une probabilité de churn calculée proche de 0.5 — zone grise.')
print('   Ajuster le seuil de décision (ex. 0.4 au lieu de 0.5) pourrait les capturer.')

Top 10 churners manqués par MLP (faux négatifs) :


,age,tenure_months,monthly_logins,weekly_active_days,avg_session_time,features_used,usage_growth_rate,last_login_days_ago,monthly_fee,total_revenue,...,signup_channel,contract_type,payment_method,discount_applied,price_increase_last_3m,complaint_type,survey_response,y_true,y_pred,churn_proba
1188,74,2,12,6,12.952057,2,0.07,4,10,20,...,Web,Monthly,PayPal,No,No,Billing,Satisfied,1,0,0.494947
1478,49,9,27,7,17.294388,3,-0.31,21,30,270,...,Web,Monthly,PayPal,No,No,Technical,Unsatisfied,1,0,0.493113
307,31,19,20,2,26.076229,5,0.11,7,10,190,...,Referral,Monthly,PayPal,Yes,No,Technical,Unsatisfied,1,0,0.491750
1838,71,38,6,6,13.037838,5,0.17,8,20,760,...,Mobile,Yearly,Bank Transfer,No,No,Service,Neutral,1,0,0.490249
1613,72,26,8,6,13.999932,5,-0.01,15,50,1300,...,Referral,Quarterly,Bank Transfer,No,Yes,Billing,Unsatisfied,1,0,0.487107
1542,24,5,2,2,13.415947,6,-0.11,0,30,150,...,Mobile,Quarterly,Card,No,No,Billing,Satisfied,1,0,0.484064
1671,39,3,0,4,9.467965,4,0.21,50,10,30,...,Web,Monthly,Card,Yes,No,Billing,Unsatisfied,1,0,0.465616
669,54,57,32,4,23.614796,6,0.18,0,70,3990,...,Web,Monthly,PayPal,No,No,Technical,Neutral,1,0,0.462528
87,70,53,2,2,3.854994,5,-0.19,10,100,5300,...,Web,Quarterly,PayPal,No,No,Technical,Satisfied,1,0,0.460793
1031,47,5,23,4,16.343543,4,-0.12,5,50,250,...,Web,Quarterly,Card,Yes,No,Billing,Satisfied,1,0,0.433482



→ Ces clients ont une probabilité de churn calculée proche de 0.5 — zone grise.
   Ajuster le seuil de décision (ex. 0.4 au lieu de 0.5) pourrait les capturer.


## 8. Ajustement du seuil de décision

In [12]:
from sklearn.metrics import f1_score, recall_score

thresholds = np.arange(0.2, 0.8, 0.05)
f1_scores  = [f1_score(y_test, (best_proba >= t).astype(int), zero_division=0) for t in thresholds]
rec_scores = [recall_score(y_test, (best_proba >= t).astype(int), zero_division=0) for t in thresholds]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores,  label='F1', marker='o')
ax.plot(thresholds, rec_scores, label='Recall', marker='s', linestyle='--')
ax.axvline(0.5, color='gray', linestyle=':', label='Seuil par défaut (0.5)')
ax.set_xlabel('Seuil de décision')
ax.set_title(f'F1 et Recall selon le seuil — {best_name}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/03_threshold_analysis.png', bbox_inches='tight')
plt.show()

best_threshold = thresholds[np.argmax(f1_scores)]
print(f'Seuil optimal (max F1) : {best_threshold:.2f}')

Seuil optimal (max F1) : 0.50


C:\Users\Pret Jeff\AppData\Local\Temp\ipykernel_10880\3115868508.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Sauvegarde du meilleur modèle

In [13]:
all_trained = {
    'Logistic Regression': lr,
    'Random Forest': rf,
    'XGBoost': xgb_model,
    'MLP': mlp,
}
best_model = all_trained[best_name]

print(f'Meilleur modèle : {best_name}')
print(f'Justification : {comparison.loc[best_name].to_dict()}')

if best_name == 'MLP':
    best_model.save('../models/best_model_churn.keras')
else:
    joblib.dump(best_model, '../models/best_model_churn.joblib')

joblib.dump(comparison, '../models/churn_comparison.joblib')

# Sauvegarder les probabilités pour Tâche B
import pandas as pd
pd.DataFrame({
    'churn_proba': best_proba,
    'total_revenue': X_test_raw['total_revenue'].values if 'total_revenue' in X_test_raw.columns else 0,
}).to_parquet('../models/churn_proba_test.parquet', index=False)

print('→ Prochaine étape : 04_modeling_revenue_risk.ipynb (Tâche B — bonus)')

Meilleur modèle : MLP
Justification : {'accuracy': 0.786, 'precision': 0.2455, 'recall': 0.5294, 'f1': 0.3354, 'roc_auc': 0.7422, 'pr_auc': 0.2188}


→ Prochaine étape : 04_modeling_revenue_risk.ipynb (Tâche B — bonus)


## Synthèse et justification du modèle retenu

| Critère | LR | RF | XGBoost | MLP ✓ |
|---|---|---|---|---|
| F1 score | 0.311 | 0.193 | 0.174 | **0.335** |
| Recall | 0.672 | 0.132 | 0.132 | **0.529** |
| PR-AUC | 0.271 | 0.271 | 0.219 | 0.219 |
| ROC-AUC | 0.748 | **0.802** | 0.741 | 0.742 |
| CV F1 (±std) | 0.320 ±0.012 | 0.156 ±0.037 | 0.192 ±0.027 | — |
| Interprétabilité | Haute | Moyenne | Moyenne | Faible |
| Déployabilité | Simple | Simple | Simple | Complexe |

**Modèle retenu : MLP**

Le MLP obtient le meilleur F1 (0.335) et surtout le meilleur **rappel (52.9%)** — il détecte plus de churners réels que les autres modèles. Dans un contexte de rétention client, un faux négatif (churner non détecté) coûte beaucoup plus cher qu'une action de rétention déclenchée à tort.

La Régression Logistique suit de très près (F1=0.311, rappel=0.67) avec une bien meilleure stabilité en CV et une interprétabilité supérieure — elle reste une alternative sérieuse pour un contexte où l'explicabilité est prioritaire.

**Pourquoi le Random Forest et XGBoost déçoivent ici ?** Leur précision est plus élevée mais leur rappel s'effondre (13%) : ils sont trop conservateurs et ratent la majorité des churners. L'équilibrage des classes via `class_weight` a moins bien fonctionné pour ces modèles sur cette distribution particulière.

**Sur le deep learning :** le MLP s'en sort légèrement mieux grâce à sa capacité à modéliser des interactions non-linéaires complexes entre features. Mais l'écart est modeste, et ce résultat ne se généralisera pas nécessairement sur d'autres datasets — la LR reste compétitive.